# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import os
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# engineered features: log-transform heavy-tailed counts, has_-flags instead
# of a blind fillna(0) (word_count/search_volume are missing along
# content_type lines -- a bare 0 would silently encode content type)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

NUMERIC = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count",
    "char_count", "search_volume", "competition", "cpc", "has_word_count", "has_keyword_data", "has_position"]
CATEGORICAL = ["content_type", "main_intent", "competition_level", "age_tier"]
FEATURES = NUMERIC + CATEGORICAL

feature_vector = df[["content_id", "client_id"] + FEATURES + ["is_declining_label"]]
print(f"Feature vector: {feature_vector.shape[0]:,} rows x {len(FEATURES)} features")
print(f"Categorical missingness handled by SimpleImputer(fill_value='unknown') downstream;")
print(f"numeric missingness by SimpleImputer(fill_value=0) + has_-flags above.")
feature_vector.head(3)

Working directory: C:\Users\Laptop\Documents\fly


Feature vector: 30,000 rows x 25 features
Categorical missingness handled by SimpleImputer(fill_value='unknown') downstream;
numeric missingness by SimpleImputer(fill_value=0) + has_-flags above.


,content_id,client_id,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,days_with_sessions,content_age_days,days_since_last_update,...,competition,cpc,has_word_count,has_keyword_data,has_position,content_type,main_intent,competition_level,age_tier,is_declining_label
0,content_304f48230142,client_f369cb89fc,8.243808,3.401197,2.890372,0.0,88,13,187,20,...,0.67,2.05,1,1,1,keyword article,transactional,HIGH,181-365,1
1,content_a1fb4e703a9e,client_4e07408562,9.636980,2.079442,2.302585,0.0,88,9,445,25,...,0.01,0.05,1,1,1,keyword article,informational,LOW,365+,1
2,content_9aa793d4d895,client_7f2253d7e2,9.440023,2.484907,2.484907,0.0,88,11,141,20,...,0.00,0.00,1,1,1,keyword article,informational,LOW,91-180,1


### 1. Build the feature vector

25 features: 21 numeric (log-transformed traffic counts, missingness-aware `has_-flags` instead
of a blind `fillna(0)`, since word_count/search_volume missingness follows `content_type`) and 4
categorical, all sourced from the starter dataset's observed, same-day columns.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
FEATURE_NOTES = {
    "log_impressions_90d":   ("log1p of 90d GSC impressions", "0 if missing (none are)", "numeric", "yes -- same-day rollup"),
    "log_clicks_90d":        ("log1p of 90d GSC clicks", "0 if missing", "numeric", "yes"),
    "log_sessions_90d":      ("log1p of 90d GA4 sessions", "0 if missing", "numeric", "yes"),
    "log_ai_sessions_90d":   ("log1p of 90d AI-referral sessions", "0 if missing", "numeric", "yes"),
    "days_with_impressions": ("days in 90d window with >=1 impression", "0 if missing", "numeric", "yes"),
    "days_with_sessions":    ("days in 90d window with >=1 session", "0 if missing", "numeric", "yes"),
    "content_age_days":      ("days since content created", "0 if missing (none are)", "numeric", "yes"),
    "days_since_last_update":("days since last content update", "0 if missing", "numeric", "yes, but see w03_data_contract's snapshot-trap finding for the warehouse equivalent"),
    "ctr":                   ("90d clicks/impressions x100", "0 if missing", "numeric", "yes"),
    "avg_position":          ("mean GSC position; 0 = no data", "0 IS a real value here (no-data code)", "numeric", "yes"),
    "engagement_rate":       ("engaged_sessions/sessions x100", "0 if missing", "numeric", "yes"),
    "scroll_rate":           ("scroll_events/pageviews x100, can exceed 100", "0 if missing", "numeric", "yes"),
    "ai_traffic_pct":        ("ai_sessions/sessions x100, can exceed 100", "0 if missing", "numeric", "yes"),
    "word_count":            ("article word count", "0 + has_word_count flag (28.3% missing, follows content_type)", "numeric", "yes"),
    "char_count":            ("article char count", "0 + implied by has_word_count", "numeric", "yes"),
    "search_volume":         ("keyword search-volume estimate", "0 + has_keyword_data flag", "numeric", "yes -- external keyword data, not page performance"),
    "competition":           ("keyword competition score 0-1", "0 if missing", "numeric", "yes"),
    "cpc":                   ("keyword cost-per-click estimate", "0 if missing", "numeric", "yes"),
    "has_word_count":        ("1 if word_count observed", "n/a (a flag)", "binary", "yes"),
    "has_keyword_data":      ("1 if search_volume observed", "n/a (a flag)", "binary", "yes"),
    "has_position":          ("1 if avg_position > 0", "n/a (a flag)", "binary", "yes"),
    "content_type":          ("keyword/feedly/comparison article", "'unknown' if missing", "categorical", "yes"),
    "main_intent":           ("informational/transactional/commercial/navigational", "'unknown' if missing", "categorical", "yes"),
    "competition_level":     ("LOW/MEDIUM/HIGH", "'unknown' if missing", "categorical", "yes"),
    "age_tier":              ("bucketed content_age_days", "'unknown' if missing (none are)", "categorical", "yes"),
}
notes_df = pd.DataFrame(FEATURE_NOTES).T
notes_df.columns = ["meaning", "missing_handling", "type", "available_before_decision_moment"]
print(notes_df.to_string())

                                                                    meaning                                               missing_handling         type                                                     available_before_decision_moment
log_impressions_90d                            log1p of 90d GSC impressions                                        0 if missing (none are)      numeric                                                               yes -- same-day rollup
log_clicks_90d                                      log1p of 90d GSC clicks                                                   0 if missing      numeric                                                                                  yes
log_sessions_90d                                  log1p of 90d GA4 sessions                                                   0 if missing      numeric                                                                                  yes
log_ai_sessions_90d                       log1p of 9

### 2. Feature notes

Every feature audited for meaning, missingness handling, type, and whether it's knowable before
the decision moment. All 25 pass the "available before" test — none require peeking at
`trend_direction`, `trend_pct`, or any future window. The one caveat worth repeating:
`days_since_last_update` is safe here because the starter CSV is a single pre-aggregated
snapshot per page (not a daily panel) — the warehouse equivalent of this same feature has a real
trap (`w03_data_contract.ipynb`'s finding that 88.1% of March rows show `dim_content`'s update
date *after* that row's own date).

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
import sys
sys.path.append("scripts")
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

SEED = 42
groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)

def eval_features(extra_numeric=None):
    num = NUMERIC + (extra_numeric or [])
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0)), ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
    ])
    X = df[num + CATEGORICAL]
    y = df["is_declining_label"]
    tr, te = next(gss.split(X, y, groups))
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])
    pipe.fit(X.iloc[tr], y.iloc[tr])
    proba = pipe.predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y.iloc[te], proba)

auc_honest = eval_features()
print(f"HONEST feature set, client-grouped holdout: AUC = {auc_honest:.3f}")

# ATTACK: deliberately smuggle the label source itself back in as a "feature"
auc_leak_trendpct = eval_features(extra_numeric=["trend_pct"])
print(f"\nATTACK 1 -- add trend_pct (the label's own source): AUC = {auc_leak_trendpct:.3f}")

# a second, subtler attack: the label's raw ingredients (not the pre-computed ratio)
auc_leak_last30 = eval_features(extra_numeric=["impressions_last_30d", "impressions_prev_30d"])
print(f"ATTACK 2 -- add impressions_last_30d/prev_30d (label's raw inputs): AUC = {auc_leak_last30:.3f}")

# smell test: does any HONEST feature correlate suspiciously with the label?
corr = df[NUMERIC].corrwith(df["is_declining_label"]).sort_values(key=lambda s: s.abs(), ascending=False)
print(f"\nMax |correlation| among honest features: {corr.abs().max():.3f} (flag threshold 0.9 -- clean)")

HONEST feature set, client-grouped holdout: AUC = 0.620



ATTACK 1 -- add trend_pct (the label's own source): AUC = 0.999


ATTACK 2 -- add impressions_last_30d/prev_30d (label's raw inputs): AUC = 0.844

Max |correlation| among honest features: 0.220 (flag threshold 0.9 -- clean)


### 3. The leakage hunt

The honest feature set holds up: AUC 0.620 on a client-grouped holdout — modest, and consistent
with how weak any single signal looked in Week 2's exploration. Both deliberate attacks behave
exactly as they should: smuggling in `trend_pct` (the label's own defining column) pushes AUC to
0.999 — not "very good," a giveaway. The subtler attack — the label's *raw* ingredients
(`impressions_last_30d`/`prev_30d`) rather than the pre-computed ratio — still inflates AUC to
0.844, a smaller but real leak, which matters because a raw ingredient is easier to mistake for a
legitimate feature than an obviously-named `trend_pct` column would be. Neither attack column
made it into the honest feature set; the correlation smell-test above confirms no *unlabeled*
feature is quietly doing the same thing (max |r| well under the 0.9 flag threshold).

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
EXCLUDED = {
    "trend_direction": "label's own source",
    "trend_pct": "label's own source (Attack 1 above)",
    "impressions_last_30d": "label's raw ingredient (Attack 2 above)",
    "impressions_prev_30d": "label's raw ingredient (Attack 2 above)",
    "clicks_last_30d": "label window, not used but same family",
    "clicks_prev_30d": "label window, not used but same family",
    "provider_used": "not observed search/engagement behavior",
    "model_used": "not observed search/engagement behavior",
    "content_id": "identifier, grouping only",
    "client_id": "identifier, grouping/splitting only",
}
overlap = set(EXCLUDED) & set(FEATURES)
print("Excluded columns that leaked into FEATURES anyway?", bool(overlap), "->", overlap or "none")
for col, reason in EXCLUDED.items():
    print(f"  {col}: {reason}")

Excluded columns that leaked into FEATURES anyway? False -> none
  trend_direction: label's own source
  trend_pct: label's own source (Attack 1 above)
  impressions_last_30d: label's raw ingredient (Attack 2 above)
  impressions_prev_30d: label's raw ingredient (Attack 2 above)
  clicks_last_30d: label window, not used but same family
  clicks_prev_30d: label window, not used but same family
  provider_used: not observed search/engagement behavior
  model_used: not observed search/engagement behavior
  content_id: identifier, grouping only
  client_id: identifier, grouping/splitting only


### 4. What I excluded and why

- `trend_direction`, `trend_pct` — the label's own source; using them as features would be
  answering the question with the question.
- `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d` — the
  label's raw ingredients; Attack 2 above shows exactly why (AUC 0.844, a real leak even without
  the exact label formula).
- `provider_used`, `model_used` — not observed search or engagement behavior; excluded by data
  dictionary convention.
- `content_id`, `client_id` — identifiers; grouping and splitting only, never features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.